In [12]:
# Import necessary libraries for data manipulation and machine learning
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load the dataset
df = pd.read_csv('hotel_bookings.csv')

# Display the first 5 rows to confirm the data loaded correctly
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [13]:
# Drop Data Leakage Columns
df = df.drop(['reservation_status', 'reservation_status_date'], axis=1)

# Handle Missing Values
# company column has too many missing values, so we drop the entire column
df = df.drop(['company'], axis=1) 

# For 'agent' and 'children', assume a missing value means 0 (no agent, no children)
df['agent'] = df['agent'].fillna(0)
df['children'] = df['children'].fillna(0)

# For 'country', filling missing values with the most frequent country (the mode)
df['country'] = df['country'].fillna(df['country'].mode()[0])

# Encode Categorical Variables
categorical_cols = df.select_dtypes(include=['object']).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Preprocessing complete. Current dataset shape:", df.shape)

Preprocessing complete. Current dataset shape: (119390, 247)


In [14]:
# Separate Features (X) and Target Variable (y/ 'is_canceled' column)

X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

# Split the data into Training and Testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Gradient Boosting Classifier

gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

print("Training the Gradient Boosting model...")

# Train the model on the training data
gb_clf.fit(X_train, y_train)

print("Model training complete!")

Training the Gradient Boosting model...
Model training complete!


In [15]:
# Make predictions using the test data
y_pred = gb_clf.predict(X_test)

# Calculate and print the overall accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

# Print the detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 85.22%

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.92      0.89     14907
           1       0.85      0.73      0.79      8971

    accuracy                           0.85     23878
   macro avg       0.85      0.83      0.84     23878
weighted avg       0.85      0.85      0.85     23878

